# Make an Adapters Directory

In [1]:
import os

if not os.path.exists('./adapters'):
    os.makedirs('./adapters')
    print("Created './adapters' directory.")
else:
    print("'./adapters' directory already exists.")

Created './adapters' directory.


# Unzip the adapter to this Adapters Directory

In [3]:
!unzip -o '/content/medical-qa-v1-adapter.zip' -d './'

Archive:  /content/medical-qa-v1-adapter.zip
   creating: ./adapters/medical-qa-v1/
  inflating: ./adapters/medical-qa-v1/adapter_model.safetensors  
  inflating: ./adapters/medical-qa-v1/tokenizer_config.json  
  inflating: ./adapters/medical-qa-v1/chat_template.jinja  
  inflating: ./adapters/medical-qa-v1/adapter_config.json  
  inflating: ./adapters/medical-qa-v1/README.md  
  inflating: ./adapters/medical-qa-v1/tokenizer.json  
  inflating: ./adapters/medical-qa-v1/training_args.bin  


# Inspect the Adapter

In [4]:
from safetensors import safe_open

with safe_open("./adapters/medical-qa-v1/adapter_model.safetensors",
               framework="pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        print(f"{key:80s} {tuple(tensor.shape)} {tensor.dtype}")

base_model.model.model.language_model.layers.0.mlp.down_proj.lora_A.weight       (4, 10240) torch.float32
base_model.model.model.language_model.layers.0.mlp.down_proj.lora_B.weight       (2560, 4) torch.float32
base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_A.weight       (4, 2560) torch.float32
base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_B.weight       (10240, 4) torch.float32
base_model.model.model.language_model.layers.0.mlp.up_proj.lora_A.weight         (4, 2560) torch.float32
base_model.model.model.language_model.layers.0.mlp.up_proj.lora_B.weight         (10240, 4) torch.float32
base_model.model.model.language_model.layers.0.per_layer_input_gate.lora_A.weight (4, 2560) torch.float32
base_model.model.model.language_model.layers.0.per_layer_input_gate.lora_B.weight (256, 4) torch.float32
base_model.model.model.language_model.layers.0.per_layer_projection.lora_A.weight (4, 256) torch.float32
base_model.model.model.language_model.layers.0.per_

# Load an adapter from HF Hub

In [ ]:
from huggingface_hub import logout, login, create_repo, upload_folder
from google.colab import userdata

# Ensure your Hugging Face token is stored in Colab secrets as 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Try to log out first to clear any old sessions
try:
    logout()
    print("Successfully logged out (or no active session).")
except Exception as e:
    print(f"Error during logout, continuing anyway: {e}")

# Log in using the explicitly fetched token
login(token=hf_token)
print("Logged in with new token.")

create_repo("lmoroney/medical-qa-v1-gemma4-lora",
            private=True, exist_ok=True)

upload_folder(
    repo_id="lmoroney/medical-qa-v1-gemma4-lora",
    folder_path="./adapters/medical-qa-v1",
    commit_message="Initial upload of medical Q&A adapter",
)

Error during logout, continuing anyway: You are automatically logged in using a Google Colab secret.
To log out, you must unset the `HF_TOKEN` secret in your Colab settings.
Logged in with new token.


CommitInfo(commit_url='https://huggingface.co/lmoroney/medical-qa-v1-gemma4-lora/commit/49289e068a347df1b63ca3bc45b79ca5139614de', commit_message='Initial upload of medical Q&A adapter', commit_description='', oid='49289e068a347df1b63ca3bc45b79ca5139614de', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lmoroney/medical-qa-v1-gemma4-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='lmoroney/medical-qa-v1-gemma4-lora'), pr_revision=None, pr_num=None)

# Load a local adapter and apply it

In [6]:
%pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Step 1: Load the base model
model_name = "google/gemma-4-E4b-it"
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Step 2: Load the tokenizer (from the adapter folder, so you pick up
# any custom tokens or chat template changes)
tokenizer = AutoTokenizer.from_pretrained("./adapters/medical-qa-v1")

# Step 3: Attach the adapter
model = PeftModel.from_pretrained(base_model, "./adapters/medical-qa-v1")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

# Load the base model, without adapter, and get answer

In [8]:
import gc
# del model, ft_model
gc.collect()
torch.cuda.empty_cache()

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "google/gemma-4-E4B-it"
adapter_path = "./adapters/medical-qa-v1"
# Or the Hub version:
# adapter_path = "lmoroney/medical-qa-v1-gemma4-lora"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto",
)

prompt = "What are the common symptoms of hypothyroidism?"

messages = [{"role": "user", "content": prompt}]
input_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=512,
        temperature=0.7, top_p=0.9, do_sample=True,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
)
print("BASE MODEL:")
print(response)

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

BASE MODEL:
Hypothyroidism, which means the thyroid gland isn't producing enough thyroid hormone, can manifest in a wide variety of ways because the thyroid hormone regulates so many bodily functions. Symptoms can be subtle and easily mistaken for other conditions, which is why regular check-ups are important.

Here is a comprehensive list of the common symptoms, categorized by the body system they affect:

---

### 🧠 Mental and Cognitive Symptoms
These are often among the earliest and most overlooked signs.
* **Fatigue and Lethargy:** Persistent, overwhelming tiredness that isn't relieved by rest.
* **Brain Fog:** Difficulty concentrating, forgetfulness, and feeling mentally slow.
* **Depression or Mood Changes:** Feeling sluggish, anxious, or experiencing persistent low moods.
* **Slowed Thinking:** A general feeling of mental slowness.

### 🌡️ Metabolic and Temperature Changes
Since the thyroid controls metabolism, these changes are very common.
* **Cold Intolerance:** Feeling cold 

# Attach adapter and run same prompt

In [11]:
# Explicitly reload — do NOT reuse a wrapped model variable
#import gc
#del model
#gc.collect()
#torch.cuda.empty_cache()

#model = AutoModelForCausalLM.from_pretrained(
#    model_name, torch_dtype=torch.bfloat16, device_map="auto",
#)
ft_model = PeftModel.from_pretrained(model, adapter_path)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = ft_model.generate(
        **inputs, max_new_tokens=512,
        temperature=0.7, top_p=0.9, do_sample=True,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
)
print("FINE-TUNED MODEL:")
print(response)

FINE-TUNED MODEL:
Common signs of hypothyroidism include weight gain without changes in diet, feeling cold more than usual, fatigue, constipation, dry skin, hair loss, slow or irregular heartbeat, muscle aches, and mood changes such as depression. In some cases, you may also notice a puffy face or swelling around the eyes. 
If you notice any of these symptoms, it’s important to consult a healthcare professional for proper testing and evaluation.


# Measure Perplexity
## Helper Functions

In [13]:
import math
import gc
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "google/gemma-4-E4b-it"
adapter_path = "./adapters/medical-qa-v1"
# Or the Hub version:
# adapter_path = "lmoroney/medical-qa-v1-gemma4-lora"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
test_dataset = load_dataset(
    "json", data_files="test_set.jsonl", split="train"
)


def compute_perplexity(model, tokenizer, dataset, max_length=1024):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for example in dataset:
        messages = [
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["answer"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        inputs = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=max_length,
        ).to(model.device)
        labels = inputs["input_ids"].clone()

        with torch.no_grad():
            outputs = model(**inputs, labels=labels)

        total_loss += outputs.loss.item() * inputs["input_ids"].numel()
        total_tokens += inputs["input_ids"].numel()

    return math.exp(total_loss / total_tokens)

Generating train split: 0 examples [00:00, ? examples/s]

# Load and measure for base model

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto",
)
base_ppl = compute_perplexity(model, tokenizer, test_dataset)
print(f"Base model perplexity: {base_ppl:.3f}")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Base model perplexity: 27.412


# Clear the model, reload, add adapter, and measure again

In [15]:
del model
gc.collect()
torch.cuda.empty_cache()

In [16]:
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto",
)
adapter_path = "./adapters/medical-qa-v1"
model = PeftModel.from_pretrained(model, adapter_path)
ft_ppl = compute_perplexity(model, tokenizer, test_dataset)
print(f"Fine-tuned model perplexity: {ft_ppl:.3f}")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Fine-tuned model perplexity: 8.446


# Compare Measurements

In [17]:
improvement = (base_ppl - ft_ppl) / base_ppl * 100
print(f"\nBase model perplexity:       {base_ppl:.3f}")
print(f"Fine-tuned model perplexity: {ft_ppl:.3f}")
print(f"Relative improvement:        {improvement:.1f}%")


Base model perplexity:       27.412
Fine-tuned model perplexity: 8.446
Relative improvement:        69.2%
